# Laboratory Task 4 — PyTorch Regression

**Instruction:** Train a regression model in PyTorch using: **MSE Loss**, **2 fully connected layers**, **batch size 8**, **SGD**, and **1000 epochs**.

The regression dataset used below is the same temperature/rainfall/humidity → apples/oranges dataset demonstrated in the source notebook.

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(42); np.random.seed(42)
x_train=np.array([[73,67,43],[91,88,64],[87,134,58],[102,43,37],[69,96,70],[73,67,43],[91,88,64],[87,134,58],[102,43,37],[69,96,70],[73,67,43],[91,88,64],[87,134,58],[102,43,37],[69,96,70]],dtype='float32')
y_train=np.array([[56,70],[81,101],[119,133],[22,37],[103,119],[56,70],[81,101],[119,133],[22,37],[103,119],[56,70],[81,101],[119,133],[22,37],[103,119]],dtype='float32')

## 1. Prepare the dataset

The inputs and targets are standardized so SGD remains numerically stable during 1000 epochs. Predictions are converted back to their original scale at the end.

In [2]:
x_mean=x_train.mean(axis=0); x_std=x_train.std(axis=0)
y_mean=y_train.mean(axis=0); y_std=y_train.std(axis=0)
X=torch.tensor((x_train-x_mean)/x_std,dtype=torch.float32)
Y=torch.tensor((y_train-y_mean)/y_std,dtype=torch.float32)
train_ds=TensorDataset(X,Y)
train_dl=DataLoader(train_ds,batch_size=8,shuffle=True)
print('Samples:',len(train_ds)); print('Batch size:',train_dl.batch_size)

Samples: 15
Batch size: 8


## 2. Define a model with two fully connected layers

Two `nn.Linear` layers are used. Because the task asks for a linear regression model, no nonlinear activation is inserted between them; the overall mapping therefore remains linear.

In [3]:
class RegressionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1=nn.Linear(3,8)
        self.fc2=nn.Linear(8,2)
    def forward(self,x):
        return self.fc2(self.fc1(x))
model=RegressionModel()
print(model)

RegressionModel(
  (fc1): Linear(in_features=3, out_features=8, bias=True)
  (fc2): Linear(in_features=8, out_features=2, bias=True)
)


## 3. MSE criterion and SGD optimizer

In [4]:
criterion=nn.MSELoss()
optimizer=torch.optim.SGD(model.parameters(),lr=0.01)
print('Criterion:',criterion); print('Optimizer: SGD')

Criterion: MSELoss()
Optimizer: SGD


## 4. Train for 1000 epochs

In [5]:
num_epochs=1000
loss_history=[]
for epoch in range(num_epochs):
    model.train(); epoch_loss=0.0
    for xb,yb in train_dl:
        pred=model(xb); loss=criterion(pred,yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        epoch_loss += loss.item()*xb.size(0)
    epoch_loss/=len(train_ds); loss_history.append(epoch_loss)
    if (epoch+1)%100==0:
        print(f'Epoch [{epoch+1:4d}/{num_epochs}], Loss: {epoch_loss:.6f}')

Epoch [ 100/1000], Loss: 0.002257
Epoch [ 200/1000], Loss: 0.000599


Epoch [ 300/1000], Loss: 0.000471


Epoch [ 400/1000], Loss: 0.000435
Epoch [ 500/1000], Loss: 0.000421


Epoch [ 600/1000], Loss: 0.000413


Epoch [ 700/1000], Loss: 0.000412
Epoch [ 800/1000], Loss: 0.000411


Epoch [ 900/1000], Loss: 0.000413


Epoch [1000/1000], Loss: 0.000411


## 5. Evaluate the trained model

In [6]:
model.eval()
with torch.no_grad():
    pred_scaled=model(X).numpy()
    pred_original=pred_scaled*y_std+y_mean
    mse_original=np.mean((pred_original-y_train)**2)
print('Final MSE on original target scale:',round(float(mse_original),6))
print('\nPredictions vs. targets (first 5):')
for i in range(5): print(f'{i+1}: prediction={np.round(pred_original[i],2)}, target={y_train[i]}')

Final MSE on original target scale: 0.483345

Predictions vs. targets (first 5):
1: prediction=[56.9  69.93], target=[56. 70.]
2: prediction=[ 82.36 100.92], target=[ 81. 101.]
3: prediction=[118.76 133.  ], target=[119. 133.]
4: prediction=[21.11 37.06], target=[22. 37.]
5: prediction=[101.86 119.09], target=[103. 119.]


## Conclusion

The model satisfies all required parameters: MSE Loss, two fully connected layers, batch size 8, SGD optimizer, and 1000 training epochs. The final cell reports the trained model’s MSE and compares predictions with the target values.